# M1.1 Vector Calculus on a Grid

Book pointer: chapter 1, sections 1.2 (vector fields), 1.3 (integration), 1.4 (differentiation), 1.5 (coordinate systems).

## 1. Recall

Answer on paper before running any cell.

1. Write the gradient, divergence and curl in Cartesian components. Which takes a scalar to a vector, which a vector to a scalar, which a vector to a vector?
2. In one sentence each: what does the divergence measure at a point, and what does the curl measure?
3. State the divergence theorem and Stokes' theorem. Which boundary appears in each?
4. Why are $\nabla \times (\nabla f) = 0$ and $\nabla \cdot (\nabla \times \mathbf{F}) = 0$ true for any smooth $f$ and $\mathbf{F}$?
5. A grid has 50 points along $x$ and 30 along $y$. What shape must an array holding a scalar field on that grid have so that `V[i, j]` sits at `(x[i], y[j])`?

## 2. Principle to algorithm

A field assigns a number or a vector to every point. On a computer we only keep the values on a grid, so the three differential operators become finite differences.

$$\frac{\partial f}{\partial x}\Big|_{i} \approx \frac{f_{i+1} - f_{i-1}}{2\,\Delta x}$$

`np.gradient` applies exactly this central difference inside the array and a one-sided difference on the edges. From one derivative routine we get all three operators:

- gradient: one derivative of a scalar per axis, stacked into a vector,
- divergence: derivative of component $k$ along axis $k$, summed,
- curl: the six cross derivatives combined with the right signs.

The grid itself is built with `np.meshgrid(x, y, indexing="ij")`. With `"ij"`, index `i` runs along `x` and index `j` along `y`, which matches how we write $f_{i,j}$ on paper. The default `"xy"` mode swaps the first two axes. Every field plotted from an `"xy"` grid with `"ij"` formulas is transposed, and the error is silent.

### Animation

A field morphs from pure rotation $(-y, x)$ to pure expansion $(x, y)$. The paddlewheel spins at a rate set by the curl and stops when the curl is gone. Watch the readout: divergence and curl trade places while the arrows barely seem to change near the centre.

In [ ]:
from emlib import show
show.animation("m1_fields", "DivergenceVersusCurl")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from emlib import grids, fields

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
# The indexing pitfall, made visible.
x = np.linspace(0, 4, 5)
y = np.linspace(0, 2, 3)

Xij, Yij = np.meshgrid(x, y, indexing="ij")
Xxy, Yxy = np.meshgrid(x, y)  # default indexing="xy"

print("ij shapes:", Xij.shape, "  P[i, j] = (x[i], y[j]) ->", (Xij[3, 1], Yij[3, 1]), "expected", (x[3], y[1]))
print("xy shapes:", Xxy.shape, "  first axis is now y, not x")

In [ ]:
# Two reference fields: pure rotation and pure expansion.
x, y, P = grids.grid2d((-2, 2), (-2, 2), 21)
hx, hy = grids.spacing(x, y)
X, Y, _ = grids.components(P)
zero = np.zeros_like(X)

F_rot = np.stack([-Y, X, zero], axis=-1)   # F1 = (-y, x, 0)
F_exp = np.stack([X, Y, zero], axis=-1)    # F2 = (x, y, 0)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, F, title in zip(axes, (F_rot, F_exp), ("F1 = (-y, x)", "F2 = (x, y)")):
    ax.quiver(X, Y, F[..., 0], F[..., 1])
    ax.set_aspect("equal"); ax.set_title(title)
plt.show()

In [ ]:
# Divergence and curl of both fields on the grid.
for name, F in (("F1 rotation", F_rot), ("F2 expansion", F_exp)):
    div = fields.divergence(F[..., :2], hx, hy)
    crl = fields.curl(F, hx, hy)[..., 2]
    print(f"{name:14s}  div: mean {div.mean():+.3f}  spread {div.std():.1e}   curl_z: mean {crl.mean():+.3f}  spread {crl.std():.1e}")

### Exercise: write the divergence yourself

Replace the body of `my_divergence` with your own central-difference implementation using `np.gradient`. The check below compares it to the library version on a field with non-constant divergence.

In [ ]:
def my_divergence(F, hx, hy):
    """Divergence of a 2D vector grid F of shape (nx, ny, 2 or 3)."""
    return fields.divergence(F[..., :2], hx, hy)  # replace this line with your own implementation


F_test = np.stack([X**2 * Y, np.sin(Y) * X, zero], axis=-1)
assert np.allclose(my_divergence(F_test, hx, hy), fields.divergence(F_test[..., :2], hx, hy))
print("divergence check passed")

### Symbolic to numeric

SymPy gives exact derivatives. Use them to check the finite-difference versions. Start from a scalar potential, form $\mathbf{E} = -\nabla V$, and take its divergence and curl symbolically.

In [ ]:
xs, ys, zs = sp.symbols("x y z", real=True)
V_sym = sp.exp(-(xs**2 + ys**2)) * xs + zs**2 / 4

E_sym = [-sp.diff(V_sym, v) for v in (xs, ys, zs)]
div_sym = sum(sp.diff(E_sym[k], v) for k, v in enumerate((xs, ys, zs)))
curl_sym = [
    sp.diff(E_sym[2], ys) - sp.diff(E_sym[1], zs),
    sp.diff(E_sym[0], zs) - sp.diff(E_sym[2], xs),
    sp.diff(E_sym[1], xs) - sp.diff(E_sym[0], ys),
]
print("curl of -grad V, simplified:", [sp.simplify(c) for c in curl_sym])
print("div of -grad V (= -laplacian V):", sp.simplify(div_sym))

V_num = sp.lambdify((xs, ys, zs), V_sym, "numpy")
div_num = sp.lambdify((xs, ys, zs), div_sym, "numpy")

In [ ]:
# Compare finite differences to the exact result on a 3D grid.
x3, y3, z3, P3 = grids.grid3d((-2, 2), (-2, 2), (-1, 1), (81, 81, 21))
h3 = grids.spacing(x3, y3, z3)
X3, Y3, Z3 = grids.components(P3)

V_grid = V_num(X3, Y3, Z3)
E_grid = -fields.gradient(V_grid, *h3)
div_grid = fields.divergence(E_grid, *h3)
curl_grid = fields.curl(E_grid, *h3)

inner = (slice(2, -2),) * 3
err = np.abs(div_grid[inner] - div_num(X3, Y3, Z3)[inner]).max()
print(f"max |div error| in the interior: {err:.2e}   (grid step {h3[0]:.3f})")
print(f"max |curl(-grad V)| in the interior: {np.abs(curl_grid[inner]).max():.2e}  (should be ~0)")

## 4. Verify

Two closed-form checks of the integral theorems in the plane.

Stokes: circulation of $\mathbf{F}_1 = (-y, x)$ around the unit circle equals the integral of $(\nabla\times\mathbf{F}_1)_z = 2$ over the disk, so $2\pi$.

Divergence theorem: outward flux of $\mathbf{F}_2 = (x, y)$ through the square $[-1,1]^2$ equals the integral of $\nabla\cdot\mathbf{F}_2 = 2$ over the square, so $8$.

In [ ]:
F1 = lambda p: np.stack([-p[:, 1], p[:, 0], np.zeros(len(p))], axis=-1)
t = np.linspace(0, 2 * np.pi, 2001)
circle = np.stack([np.cos(t), np.sin(t), 0 * t], axis=-1)
circ = fields.line_integral(F1, circle)
print(f"circulation of F1 around unit circle: {circ:.6f}   exact 2*pi = {2*np.pi:.6f}")

# Outward flux of F2 through the unit square: four straight edges, F . n integrated along each.
s = np.linspace(-1, 1, 2001)
ds = s[1] - s[0]
flux = 0.0
for (px, py, nx_, ny_) in ((1, None, 1, 0), (-1, None, -1, 0), (None, 1, 0, 1), (None, -1, 0, -1)):
    xe = np.full_like(s, px) if px is not None else s
    ye = np.full_like(s, py) if py is not None else s
    Fn = xe * nx_ + ye * ny_
    flux += np.trapezoid(Fn, dx=ds)
print(f"flux of F2 out of unit square: {flux:.6f}   exact = 8")

## 5. Explore and rebuild

Explore. Build $\mathbf{F}_3 = (-y, x)/(x^2+y^2)$ on a grid that avoids the origin. Its curl is zero everywhere you can evaluate it, yet its circulation around the origin is $2\pi$. Compute both and explain where Stokes' theorem fails and why.

Rebuild. Close this notebook. On a blank editor, write `gradient`, `divergence`, `curl` and `line_integral` from scratch and make them pass `tests/test_fields.py`.